# LangChain 기반 맞춤형 콘서트 예매 가이드

NOL Ticket 상품 URL과 사용자 질문을 입력받아 HTML과 상세 이미지 OCR 원문을 수집하고, LangChain RAG 파이프라인으로 질문에 필요한 예매 정보만 구조화하여 답한다.

## 0. 패키지 설치

Colab 런타임마다 검증된 버전을 설치한다. 설치가 끝나면 `런타임 → 세션 다시 시작`을 한 번 실행한 뒤 1번 셀부터 계속한다.

In [12]:
%pip install -q \
    "numpy==2.2.6" \
    "paddlepaddle==3.2.0" \
    "paddleocr==3.7.0" \
    "pillow==11.3.0" \
    "requests==2.34.2" \
    "beautifulsoup4==4.15.0" \
    "brotli==1.2.0" \
    "python-dotenv" \
    "langchain==1.4.2" \
    "langchain-core==1.6.3" \
    "langchain-openai==1.6.2" \
    "openai==3.16.2" \
    "httpx2==2.13.0" \
    "langchain-chroma==1.1.0" \
    "langchain-text-splitters==1.1.2"

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] 액세스가 거부되었습니다: 'C:\\Users\\175072\\AppData\\Local\\Temp\\pip-uninstall-mb4mu9x7\\_brotli.cp311-win_amd64.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. 환경변수와 Chat Model

현재 폴더의 `.env`에서 `OPENAI_API_KEY`를 읽는다. 예매 정보는 일관성과 근거성이 중요하므로 `temperature=0`으로 설정한다.

In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)
PADDLE_CACHE = Path('/content/paddlex_cache')
os.environ.setdefault('PADDLE_PDX_CACHE_HOME', str(PADDLE_CACHE))
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('.env 파일에 OPENAI_API_KEY를 설정해 주세요.')

model = init_chat_model(
    'gpt-4o-mini',
    model_provider='openai',
    temperature=0,
    timeout=30,
    max_tokens=700,
    max_retries=1,
)
print('Paddle 모델 캐시:', PADDLE_CACHE)
print('Chat Model 초기화 완료:', model.model_name)

Paddle 모델 캐시: \content\paddlex_cache
Chat Model 초기화 완료: gpt-4o-mini


## 1-1. 실행 환경 검증

설치 셀에서 지정한 버전이 실제 Colab 런타임에 적용됐는지 확인한다. 버전이 다르면 0번 셀 실행 후 세션을 다시 시작해야 한다.

In [1]:
import sys
from importlib.metadata import version

import numpy
import paddle
import paddleocr

expected_versions = {
    'numpy': '2.2.6',
    'paddle': '3.2.0',
    'paddleocr': '3.7.0',
    'brotli': '1.2.0',
    'openai': '3.16.2',
    'httpx2': '2.13.0',
}
actual_versions = {
    'python': sys.version.split()[0],
    'numpy': numpy.__version__,
    'paddle': paddle.__version__,
    'paddleocr': paddleocr.__version__,
    'brotli': version('brotli'),
    'openai': version('openai'),
    'httpx2': version('httpx2'),
}

print('실행 Python:', sys.executable)
print('환경 정보:', actual_versions)

for package, expected_version in expected_versions.items():
    actual_version = actual_versions[package]
    if actual_version != expected_version:
        raise RuntimeError(
            f'{package} 버전 불일치: '
            f'expected={expected_version}, actual={actual_version}. '
            '0번 셀 실행 후 세션을 다시 시작해 주세요.'
        )

paddle.utils.run_check()

c:\Users\175072\AppData\Local\Programs\Python\Python311\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
c:\Users\175072\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


실행 Python: c:\Users\175072\AppData\Local\Programs\Python\Python311\python.exe
환경 정보: {'python': '3.11.0', 'numpy': '2.2.6', 'paddle': '3.2.0', 'paddleocr': '3.7.0', 'brotli': '1.2.0', 'openai': '3.16.2', 'httpx2': '2.13.0'}
Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


c:\Users\175072\AppData\Local\Programs\Python\Python311\Lib\site-packages\paddle\pir\math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


# 2. 사용자 입력과 URL 검증

특정 상품 ID를 코드에 저장하지 않고 실행할 때 NOL Ticket 상품 URL을 입력한다. 현재 MVP는 `https://nol.yanolja.com/ticket/products/{concert_id}` 형식만 지원한다.

예제 링크
- https://nol.yanolja.com/ticket/products/26013161
- https://nol.yanolja.com/ticket/products/26013132 : 이미지 0장
- https://nol.yanolja.com/ticket/products/26012624
- https://nol.yanolja.com/ticket/products/26012865

일단 4개 정도 예시 가져왔어..

In [2]:
import re
from urllib.parse import urlparse

PRODUCT_PATH = re.compile(r'^/ticket/products/(?P<concert_id>[0-9]+)/?$')

def extract_concert_id(url):
    parsed = urlparse(url.strip())
    if parsed.scheme != 'https' or parsed.hostname != 'nol.yanolja.com':
        raise ValueError('현재는 NOL Ticket 상품 페이지만 지원합니다.')
    match = PRODUCT_PATH.fullmatch(parsed.path)
    if match is None:
        raise ValueError('NOL Ticket 상품 URL 형식이 올바르지 않습니다.')
    return match.group('concert_id')

page_url = input('NOL Ticket 상품 URL: ').strip()
concert_id = extract_concert_id(page_url)
print('concert_id:', concert_id)

concert_id: 26012865


# 3. HTML Loader

`requests + BeautifulSoup`으로 상품 페이지를 읽는다. 공연마다 달라지는 장소·가격·팬클럽명으로 문장을 선별하지 않고 정제된 HTML 원문 전체를 보존한다. 상세 공지 이미지는 NOL Ticket이 사용하는 이미지 호스트와 경로를 기준으로 탐지한다.

In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

HEADERS = {'User-Agent': 'Mozilla/5.0', 'Accept-Language': 'ko-KR,ko;q=0.9'}

response = requests.get(page_url, headers=HEADERS, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
for tag in soup(['script', 'style', 'noscript', 'template']):
    tag.decompose()

lines = [line.strip() for line in soup.get_text('\n').splitlines() if line.strip()]
html_text = '\n'.join(dict.fromkeys(lines))
if not html_text:
    raise RuntimeError('페이지에서 공연 정보를 가져올 수 없습니다.')

IMAGE_ATTRIBUTES = ('src', 'data-src', 'data-original', 'data-lazy-src')
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.gif')
POSTER_PATH_PREFIXES = (
    '/play/image/large/',
    '/play/image/small/',
    '/ticketimage/notice_poster/',
)

def is_detail_image(image_url):
    parsed = urlparse(image_url)
    host = (parsed.hostname or '').lower()
    path = parsed.path.lower()

    if parsed.scheme not in {'http', 'https'}:
        return False
    if host != 'ticketimage.interpark.com':
        return False
    if path.startswith(POSTER_PATH_PREFIXES):
        return False
    return path.endswith(IMAGE_EXTENSIONS)

all_image_urls = []
detail_image_urls = []

for image in soup.find_all('img'):
    for attribute in IMAGE_ATTRIBUTES:
        candidate = image.get(attribute)
        if not candidate:
            continue

        image_url = urljoin(response.url, candidate.strip())
        if image_url not in all_image_urls:
            all_image_urls.append(image_url)
        if is_detail_image(image_url) and image_url not in detail_image_urls:
            detail_image_urls.append(image_url)

print('HTTP 상태:', response.status_code)
print('HTML 원문 길이:', len(html_text))
print('전체 이미지 URL 수:', len(all_image_urls))
for image_url in all_image_urls:
    status = 'OCR 대상' if image_url in detail_image_urls else '제외'
    print(f'- [{status}] {image_url}')
print('상세 이미지 수:', len(detail_image_urls))
if not detail_image_urls:
    print('상세 공지 이미지가 없어 HTML 정보만 사용합니다.')

HTTP 상태: 200
HTML 원문 길이: 2991
전체 이미지 URL 수: 5
- [제외] https://ticketimage.interpark.com/Play/image/large/26/26012865_p.gif
- [OCR 대상] https://ticketimage.interpark.com/Play/ITM/Data/Modify/2026/9/2026091410101460.jpg
- [OCR 대상] https://ticketimage.interpark.com/Play/ITM/Data/Modify/2026/9/2026090810184022.jpg
- [제외] https://static.toss.im/illusts/interpark_facepass_guideline_kr.png
- [제외] https://ticketimage.interpark.com/Play/image/large/26/26012849_p.gif
상세 이미지 수: 2


# 4. 상세 이미지 OCR

PaddleOCR의 한국어 PP-OCRv5 모델로 각 상세 이미지의 텍스트를 추출한다. OCR 원문은 날짜·시간·가격을 임의로 보정하지 않는다. 이미지가 없거나 일부 OCR이 실패해도 HTML 처리는 계속한다.

In [4]:
import json
from io import BytesIO

import numpy as np
from paddleocr import PaddleOCR
from PIL import Image

ocr = PaddleOCR(
    lang='korean',
    ocr_version='PP-OCRv5',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

def collect_ocr_texts(results):
    texts = []
    for result in results:
        payload = getattr(result, 'json', result)
        payload = payload() if callable(payload) else payload
        payload = json.loads(payload) if isinstance(payload, str) else payload
        content = payload.get('res', payload) if isinstance(payload, dict) else {}
        recognized = content.get('rec_texts', []) if isinstance(content, dict) else []
        texts.extend(text for text in recognized if isinstance(text, str) and text.strip())
    return texts

def extract_image_text(image_url):
    image_response = requests.get(image_url, headers=HEADERS, timeout=30)
    image_response.raise_for_status()
    with Image.open(BytesIO(image_response.content)) as source:
        image_array = np.asarray(source.convert('RGB'))
    texts = collect_ocr_texts(ocr.predict(image_array))
    if not texts:
        raise RuntimeError('상세 이미지에서 텍스트를 찾지 못했습니다.')
    return '\n'.join(texts)

image_texts = {}
ocr_warnings = []
for image_url in detail_image_urls:
    try:
        image_texts[image_url] = extract_image_text(image_url)
    except Exception as error:
        ocr_warnings.append(f'{image_url}: {error}')

print('OCR 성공 이미지 수:', len(image_texts))
print('OCR 실패 이미지 수:', len(ocr_warnings))

Creating model: ('PP-OCRv5_server_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `C:\Users\175072\.paddlex\official_models\PP-OCRv5_server_det`.
Fetching 6 files: 100%|██████████| 6/6 [00:05<00:00,  1.20it/s]
Creating model: ('korean_PP-OCRv5_mobile_rec', None, None)
Using official model (korean_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `C:\Users\175072\.paddlex\official_models\korean_PP-OCRv5_mobile_rec`.
Fetching 6 files: 100%|██████████| 6/6 [00:02<00:00,  2.50it/s]
Resized image size (750x16726) exceeds max_side_limit of 4000. Resizing to fit within limit.


OCR 성공 이미지 수: 2
OCR 실패 이미지 수: 0


# 5. LangChain Document

HTML과 OCR 텍스트를 각각 `Document`로 만들고 `concert_id`, `source_type`, `source_url`, `section` metadata를 보존한다.

In [5]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=html_text,
        metadata={
            'concert_id': concert_id,
            'source_type': 'html',
            'source_url': response.url,
            'section': 'page',
        },
    )
]

for image_url in detail_image_urls:
    if image_url in image_texts:
        documents.append(Document(
            page_content=image_texts[image_url],
            metadata={
                'concert_id': concert_id,
                'source_type': 'image',
                'source_url': image_url,
                'section': 'detail_notice',
            },
        ))

print('Document 수:', len(documents))
print('source_type:', [document.metadata['source_type'] for document in documents])

Document 수: 3
source_type: ['html', 'image', 'image']


# 6. Text Splitter

`RecursiveCharacterTextSplitter`로 긴 공지를 검색 가능한 Chunk로 분할한다. 분할된 Chunk에도 원본 metadata가 유지된다.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=['\n\n', '\n[', '\n※', '\n- ', '\n', '. ', ' ', ''],
    keep_separator='start',
)
chunks = splitter.split_documents(documents)

print(f'Document {len(documents)}개 → Chunk {len(chunks)}개')
print('metadata 보존:', all('concert_id' in chunk.metadata for chunk in chunks))

Document 3개 → Chunk 23개
metadata 보존: True


# 7. OpenAI Embedding + Chroma

`text-embedding-3-small`로 Chunk를 벡터화하고 Chroma에 저장한다. 모든 검색에는 현재 URL에서 추출한 `concert_id` filter를 적용하여 다른 공연의 정보가 섞이지 않게 한다.

In [9]:
import chromadb
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma(
    client=chromadb.EphemeralClient(),
    collection_name='concert_ticket_guide_notebook',
    embedding_function=embeddings,
)
vector_store.add_documents(chunks)

print('저장된 Chunk 수:', vector_store._collection.count())

저장된 Chunk 수: 23


# 8. Query Understanding + Structured Output

사용자 질문을 입력받고, Chat Model의 Structured Output으로 예매 유형·수령 방식·필요 주제를 분석한다. 질문에 명시되지 않은 조건은 만들지 않는다.

In [14]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class QueryAnalysis(BaseModel):
    booking_type: Literal['fanclub_presale', 'general_sale', 'unknown'] = 'unknown'
    ticket_delivery: Literal['delivery', 'onsite', 'unknown'] = 'unknown'
    needed_topics: list[str] = Field(default_factory=list)
    search_query: str

query_prompt = ChatPromptTemplate.from_messages([
    ('system', '콘서트 예매 질문을 검색 조건으로 구조화하세요. 질문에 명시된 조건만 사용하고 날짜, 시간, 가격을 만들지 마세요. search_query에는 검색에 필요한 핵심 한국어 키워드를 작성하세요.'),
    ('human', '{question}'),
])
query_chain = query_prompt | model.with_structured_output(QueryAnalysis)

question = input('예매 관련 질문: ').strip()
if not question:
    raise ValueError('사용자 질문을 입력해 주세요.')

query_analysis = query_chain.invoke({'question': question})
print(query_analysis.model_dump_json(indent=2))

{
  "booking_type": "unknown",
  "ticket_delivery": "unknown",
  "needed_topics": [
    "티켓 등급",
    "가격"
  ],
  "search_query": "티켓 등급별 가격"
}


# 9. Retriever

벡터 유사도 후보와 핵심어가 직접 포함된 후보를 함께 수집한 뒤 재정렬한다. HTML 근거를 우선하고 OCR Chunk가 결과를 독점하지 않도록 제한하며, 각 결과의 점수와 일치 핵심어를 출력한다.

In [15]:
import math
import re

VECTOR_CANDIDATE_K = min(10, len(chunks))
FINAL_RESULT_K = 5
MAX_IMAGE_RESULTS = 2
SEARCH_STOPWORDS = {
    '공연', '티켓', '안내', '정보', '관련', '방법', '일정',
    '언제', '어떻게', '알려줘', '해줘', '하는', '되어', '있어',
}
KOREAN_SUFFIXES = (
    '으로부터', '에서부터', '까지는', '부터는', '에서는',
    '으로', '에서', '부터', '까지', '하고',
    '은', '는', '이', '가', '을', '를', '의', '에', '로',
)

def document_key(document):
    return (
        document.metadata.get('source_url', ''),
        document.page_content,
    )

def extract_search_terms(*texts):
    terms = set()
    for text in texts:
        for token in re.findall(r'[가-힣A-Za-z0-9]+', text.lower()):
            candidates = {token}
            for suffix in KOREAN_SUFFIXES:
                if token.endswith(suffix) and len(token) > len(suffix) + 1:
                    candidates.add(token[:-len(suffix)])
            terms.update(
                candidate for candidate in candidates
                if len(candidate) >= 2 and candidate not in SEARCH_STOPWORDS
            )
    return sorted(terms)

search_query = query_analysis.search_query.strip() or question
topic_text = ' '.join(query_analysis.needed_topics)
search_terms = extract_search_terms(question, search_query, topic_text)

semantic_results = vector_store.similarity_search_with_score(
    search_query,
    k=VECTOR_CANDIDATE_K,
    filter={'concert_id': concert_id},
)
semantic_ranks = {
    document_key(document): rank
    for rank, (document, _) in enumerate(semantic_results, start=1)
}

document_frequency = {
    term: sum(term in chunk.page_content.lower() for chunk in chunks)
    for term in search_terms
}
lexical_scores = {}
matched_terms = {}
for chunk in chunks:
    key = document_key(chunk)
    content = chunk.page_content.lower()
    matches = [term for term in search_terms if term in content]
    matched_terms[key] = matches
    lexical_scores[key] = sum(
        math.log((len(chunks) + 1) / (document_frequency[term] + 1)) + 1
        for term in matches
    )

max_lexical_score = max(lexical_scores.values(), default=0)
candidate_documents = {document_key(document): document for document, _ in semantic_results}
candidate_documents.update({
    document_key(chunk): chunk
    for chunk in chunks
    if lexical_scores[document_key(chunk)] > 0
})

ranked_candidates = []
for key, document in candidate_documents.items():
    semantic_rank = semantic_ranks.get(key)
    semantic_score = (
        (VECTOR_CANDIDATE_K - semantic_rank + 1) / VECTOR_CANDIDATE_K
        if semantic_rank is not None else 0
    )
    lexical_score = (
        lexical_scores.get(key, 0) / max_lexical_score
        if max_lexical_score else 0
    )
    html_bonus = (
        0.10
        if document.metadata.get('source_type') == 'html' and lexical_score > 0
        else 0
    )
    combined_score = 0.35 * semantic_score + 0.55 * lexical_score + html_bonus
    ranked_candidates.append((combined_score, semantic_rank, document))

ranked_candidates.sort(
    key=lambda item: (
        -item[0],
        item[2].metadata.get('source_type') != 'html',
        item[1] if item[1] is not None else VECTOR_CANDIDATE_K + 1,
    )
)

retrieved_documents = []
retrieval_debug = {}
image_result_count = 0
for combined_score, semantic_rank, document in ranked_candidates:
    if document.metadata.get('source_type') == 'image':
        if image_result_count >= MAX_IMAGE_RESULTS:
            continue
        image_result_count += 1
    retrieved_documents.append(document)
    retrieval_debug[document_key(document)] = {
        'score': combined_score,
        'vector_rank': semantic_rank,
        'matched_terms': matched_terms.get(document_key(document), []),
    }
    if len(retrieved_documents) == FINAL_RESULT_K:
        break

print('벡터 검색어:', search_query)
print('핵심어:', search_terms)
for index, document in enumerate(retrieved_documents, start=1):
    debug = retrieval_debug[document_key(document)]
    print(
        f"[{index}] score={debug['score']:.3f} | "
        f"vector_rank={debug['vector_rank']} | "
        f"keywords={debug['matched_terms']} | "
        f"{document.metadata['source_type']} | {document.metadata['source_url']}"
    )
    print(document.page_content[:300].replace('\n', ' '), '\n')

검색어: 티켓 등급별 가격이 어떻게 돼? 티켓 등급별 가격
[1] image | https://ticketimage.interpark.com/Play/ITM/Data/Modify/2026/9/2026091410101460.jpg
- 티켓 예매 시 티켓 수령 방법은'모바일티켓'으로만 선택 가능합니다 -모바일티켓은 NOL 고객센터를 통한 예매가 불가합니다.NOL 웹페이지 또는 모바일 앱을 ·모바일티켓은 단말기에서 이미지로 다운로드하여 기념티켓으로 소장할 수 있습니다. -모바일티켓은 안드로이드 버전9.0 이상,i0S 버전 16.0 이상의 기기에서 사용이 가능합니다. 안되며,이로 인해 입장 시 문제가 발생할 경우 공연 주최·주관사 및 예매처는 책임지지 않습니다. -예매를 취소 

[2] html | https://nol.yanolja.com/ticket/products/26012865
[휠체어석 예매 안내] - 휠체어석 구매는 2026년 9월 18일(금) 오전 10시(KST)부터 NOL 고객센터(1544-1555 / 운영시간 오전 9시~오후 6시)를 통해 전화예매만 가능합니다. - 공연 당일 장애인 등록증(또는 복지카드)과 본인 신분증(실물)확인 후 티켓 수령이 이루어지며, 미 지참 시 예매자 및 동반 1인도 입장이 불가합니다. - 휠체어석 예매티켓은 현장수령만 가능하며 구매자 본인 티켓 수령을 원칙으로 하고 있습니다. (* 

[3] html | https://nol.yanolja.com/ticket/products/26012865
※ 본 공연은 예매대기서비스와 동일좌석 재예매서비스 이용이 제한됩니다. ※ 본 공연은 안심예매 서비스를 적용합니다. 예매에 참고 바랍니다. 상품 상세 SOUND CHECK 220,000 원 M&G 253,000 일반석 165,000 특정 기간, 특정 공연일에만 판매하는 가격이 있으니 예매 전 [가격 전체 보기]를 눌러 확인해 주세요. 가격 전체보기 운영 시간 2026년10월 17일(토) 6PM(KST) 2026년 10월 18일(일) 5P

# 10. Prompt Template + RAG Answer Chain

검색된 Context만 공연 정보의 근거로 사용한다. HTML과 OCR에 같은 정보가 있으면 HTML을 우선하며, Context에서 확인할 수 없는 날짜·시간·가격·정책은 생성하지 않는다. 결과는 Pydantic Structured Output으로 반환한다.

In [12]:
class TicketGuideResponse(BaseModel):
    summary: str
    schedule: list[str] = Field(default_factory=list)
    requirements: list[str] = Field(default_factory=list)
    ticket_info: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)

answer_prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 콘서트 예매 안내 도우미입니다.
REFERENCE DOCUMENTS에 있는 원문만 공연 정보의 근거로 사용하세요.
HTML과 OCR에 동일한 정보가 있으면 source_type이 html인 원문을 우선하세요.
Context에 없는 날짜, 시간, 가격, 정책은 추측하거나 수정하지 마세요.
질문과 관련된 근거가 없으면 summary를 정확히 \"예매 페이지에서 확인할 수 없습니다.\"로 쓰고 나머지 목록은 비우세요.
질문과 직접 관련 없는 정보는 과도하게 출력하지 마세요.
sources에는 답변에 실제 사용한 source_url만 넣으세요.'''),
    ('human', '[USER QUESTION]\n{question}\n\n[USER CONTEXT]\n{user_context}\n\n[REFERENCE DOCUMENTS]\n{context}'),
])
answer_chain = answer_prompt | model.with_structured_output(TicketGuideResponse)

def format_context(found_documents):
    ordered = sorted(found_documents, key=lambda doc: doc.metadata['source_type'] != 'html')
    return '\n\n'.join(
        f"source_type: {doc.metadata['source_type']}\n"
        f"source_url: {doc.metadata['source_url']}\n"
        f"section: {doc.metadata['section']}\n"
        f"content: {doc.page_content}"
        for doc in ordered
    )

context = format_context(retrieved_documents)
result = answer_chain.invoke({
    'question': question,
    'user_context': query_analysis.model_dump_json(),
    'context': context,
})

# 11. 최종 결과

In [13]:
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))

if ocr_warnings:
    print('\n주의: 상세 공지 이미지 분석에 실패해 일부 안내가 누락되었을 수 있습니다.')

{
  "summary": "예매 페이지에서 확인할 수 없습니다.",
  "schedule": [],
  "requirements": [],
  "ticket_info": [],
  "warnings": [],
  "sources": []
}


# 12. 이 노트북에서 확인할 수 있는 LangChain 기능

- `Document`: HTML/OCR 원문과 source metadata 관리
- `RecursiveCharacterTextSplitter`: 긴 공지를 검색 가능한 Chunk로 분할
- `OpenAIEmbeddings`: 공연 공지를 의미 기반 Vector로 변환
- `Chroma`와 `Retriever`: 현재 `concert_id`의 관련 Context만 검색
- `ChatPromptTemplate`: 질문 분석 및 근거 기반 답변 Prompt 구성
- LCEL `prompt | model`: LangChain 구성 요소 연결
- `with_structured_output`: 질문 분석과 최종 응답을 Pydantic 객체로 반환

다른 공연을 실험할 때는 URL 입력 셀부터 전체 수집·인덱싱 단계를 다시 실행한다. 같은 공연에서 질문만 바꿀 때는 Query Understanding 셀부터 다시 실행하면 된다.